## Diamond Problem

- The **diamond problem** occurs in a multiple/hybrid inheritance structure where:
  - One parent class is inherited by two child classes (**hierarchical inheritance**).
  - Another child class inherits from both of those child classes (**multiple inheritance**).
  - This creates a diamond-shaped inheritance structure.

```text
             Parent
            /      \
           ↓        ↓
       Child 1   Child 2
           \        /
            ↓      ↓
             Child 3
```

### Types involved

- `Parent → Child 1, Child 2` → **Hierarchical inheritance**
- `Child 3 → Child 1, Child 2` → **Multiple inheritance**
- Combination of both → **Hybrid inheritance**

### Why is it called the Diamond Problem?

- `Child 3` can reach `Parent` through **two different inheritance paths**:

```text
Child 3 → Child 1 → Parent
Child 3 → Child 2 → Parent
```

- If the classes contain the same method, there can be ambiguity about **which implementation should be used**.
- Python solves this method-resolution problem using **MRO (Method Resolution Order)**.

### MRO

- MRO determines the order in which Python searches for a method or attribute.

Example:

```python
class Parent:
    def work(self):
        print("Parent work")


class Child1(Parent):
    def work(self):
        print("Child 1 work")


class Child2(Parent):
    def work(self):
        print("Child 2 work")


class Child3(Child1, Child2):
    pass


obj = Child3()

obj.work()

print(Child3.mro())
```

For this example, the MRO is:

```text
Child3 → Child1 → Child2 → Parent → object
```

Therefore, when `obj.work()` is called, Python finds `work()` in `Child1` first.
### MRO - Method resolution order.

- MRO is the **order in which Python searches for attributes and methods** in a class hierarchy.
- It determines which method or attribute Python will find first when a derived class inherits from one or multiple base classes.
- It is especially important when a class inherits from **multiple classes**, because there can be multiple possible paths to find a method or attribute.


In [8]:
class A:
    def display(self):
        print("display from A class")
class B(A):
    pass
    # def display(self):
    #     print("display from B class")
class C(A):
    def show(self):
        print("Hi from C class")
    def display(self):
        print("display from c class")

class D(B,C):
    pass
    # def display(self):
    #     print("display from D class")
d1=D()
d1.display()
print(D.__mro__)


display from c class
(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)


### How MRO and C3 Linearization Work

- Suppose a child class `D` inherits from two parent classes `B` and `C`.
- Both `B` and `C` inherit from the same parent class `A`.

```text
        A
       / \
      B   C
       \ /
        D
```

- When we call a method such as `display()` on `D`, Python first checks whether `D` has that method.
- If `display()` is not present in `D`, Python searches according to the **MRO (Method Resolution Order)**.

For this example:

```text
D → B → C → A → object
```

- First, Python searches in `D`.
- If `display()` is not present, it searches in `B`.
- If it is not present in `B`, Python does **not immediately go deeper into `A`**, even though `B` inherits from `A`.
- It next searches in `C`.
- If `display()` is found in `C`, Python uses `C.display()`.
- If it is not found in `C`, Python continues to `A`, and then `object`.

### C3 Linearization

- **C3 Linearization** creates this single ordered sequence:

```text
D → B → C → A → object
```

- Python then follows this order when searching for methods and attributes.

**Remember:**

> C3 Linearization creates the order → MRO represents that order → Python searches according to that order.

So it does **not** first search deeply through `B → A` and then move to `C`. It follows the **linear MRO order**.

### Linearization

- **Linearization** converts a complex inheritance hierarchy into a **single ordered sequence of classes**.
- Python uses **C3 Linearization** to create the MRO.
- C3 Linearization = how Python calculates the order.
- MRO = the order that Python gets and uses.

In [9]:
class A:
    pass
class B:
    pass
class C:
    pass
class D:
    pass
class E:
    pass
class F(A,B,C):
    pass
class G(D,B,E):
    pass
class H(D,A):
    pass
class Z(F,G,H):
    pass
z1=Z()
print(Z.__mro__)

(<class '__main__.Z'>, <class '__main__.F'>, <class '__main__.G'>, <class '__main__.H'>, <class '__main__.D'>, <class '__main__.A'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.E'>, <class 'object'>)


### C3 Linearization calculation / Algorithm
#### Example

##### Classes

```python
class A:
    pass

class B:
    pass

class C:
    pass

class D:
    pass

class E:
    pass

class F(A, B, C):
    pass

class G(D, B, E):
    pass

class H(D, A):
    pass

class Z(F, G, H):
    pass
```
- **Head** = first element of each list; **Tail** = all remaining elements. If only one element exists, it is the head and the tail is empty.
- Take the head of the first list. If it **does not appear in the tail of any other list**, it is a valid candidate, so take it and continue.
- If the head appears in another list's **tail**, skip that list and check the next list's head. If it is valid, take it and repeat.
- After taking a head, the next element in that same list becomes the new head, and the process continues until all lists are merged.
---

#### Calculate `L(F)`

```text
L(F) = F + merge(L(A), L(B), L(C), ABC)

     = F + merge(A, B, C, ABC)

     = FA + merge(B, C, BC)

     = FAB + merge(C, C)

     = FABC
```

Therefore:

```text
L(F) = F A B C
```

---

#### Calculate `L(G)`

```text
L(G) = G + merge(L(D), L(B), L(E), DBE)

     = G + merge(D, B, E, DBE)

     = GD + merge(B, E, BE)

     = GDB + merge(E, E)

     = GDBE
```

Therefore:

```text
L(G) = G D B E
```

---

#### Calculate `L(H)`

```text
L(H) = H + merge(L(D), L(A), DA)

     = H + merge(D, A, DA)

     = HD + merge(A, A)

     = HDA
```

Therefore:

```text
L(H) = H D A
```

---

#### Calculate `L(Z)`

```text
L(Z) = Z + merge(L(F), L(G), L(H), FGH)
```

Substitute the calculated linearizations:

```text
L(Z) = Z + merge(FABCO, GDBEO, HDAO, FGH)
```

#### Step 1

```text
= ZF + merge(ABCO, GDBEO, HDAO, GH)
```

#### Step 2

```text
= ZFG + merge(ABCO, DBEO, HDAO, H)
```

#### Step 3

```text
= ZFGH + merge(ABCO, DBEO, DAO)
```

#### Step 4

```text
= ZFGHD + merge(ABCO, BEO, AO)
```

#### Step 5

```text
= ZFGHDA + merge(BCO, BEO, O)
```

#### Step 6

```text
= ZFGHDAB + merge(CO, EO, O)
```

#### Step 7

```text
= ZFGHDABC + merge(O, EO, O)
```

#### Step 8

```text
= ZFGHDABCE + merge(O, O)
```

#### Final

```text
L(Z) = Z F G H D A B C E O
```

Therefore, the **MRO of `Z`** is:

```text
Z → F → G → H → D → A → B → C → E → object
```
